# Sensitivity Tests

These sensitivity tests are intended to evaluate the response of the TM-1.7 truck model to the addition large logistics developments that are likely to be built in the near future.

The analysis focuses on three logistics projects located within the Bay Area region. These projects were selected based on their size, development status, and potential to generate substantial new employment.

| Project                             | Likelihood             | Size (Sqft) | Expected New Employment | Sqft/Employee | Location                              |
|-------------------------------------|------------------------|------------:|-------------------------:|--------------:|---------------------------------------|
| Bridgehead Industrial               | Approved in March 2026 | 3.1M        | 3,500                    | 886           | Oakley, Contra Costa – TAZ: 1182      |
| Giovannoni Logistic Center Project  | Likely                 | 2.4M        | 3,643                    | 659           | American Canyon, Napa – TAZ: 1292     |
| Suisun Logistics                    | Under Review           | 2.1M        | 2,059                    | 1,020         | Suisun City, Solano – TAZ: 1251       |

## References

- **Bridgehead Industrial:** https://ceqanet.lci.ca.gov/2024050471/4
- **Giovannoni Logistic Center Project:** https://ceqanet.lci.ca.gov/2021010104/2
- **Suisun Logistics:** https://ceqanet.lci.ca.gov/2021010044/4

## Methodology

In this notebook, we create a modified version of the `TazData.bdf` file, one of the key inputs to TM-1.7. For each project, the estimated new employment is allocated to the corresponding TAZ and distributed proportionally across existing employment in the following sectors:

```python
['RETEMPN', 'FPSEMPN', 'AGREMPN', 'MWTEMPN']
```

In [ ]:
!! pip install dbf

In [ ]:
import pandas as pd
from pathlib import Path
from dbfread import DBF
import dbf
import shutil
import numpy as np

In [ ]:
def save_df_as_dbf(df, output_path, template_dbf):
    "Saves a dataframe in dbf format to the given output_path and template_dbf"
    # Read schema from original DBF
    template = dbf.Table(str(template_dbf))
    template.open()

    field_specs = template.structure()

    template.close()

    # Create new DBF with same schema
    table = dbf.Table(str(output_path), field_specs)
    table.open(mode=dbf.READ_WRITE)

    try:
        for _, row in df.iterrows():

            values = []

            for v in row:

                if pd.isna(v):
                    values.append(None)

                # numpy types -> python types
                elif hasattr(v, "item"):
                    values.append(v.item())

                else:
                    values.append(v)

            table.append(tuple(values))

    finally:
        table.close()

In [ ]:
# Read Original TAZ file
path = "../data/external/mtc/2023_TM161_IPA_35/landuse/tazData.dbf"
taz = pd.DataFrame(DBF(path))

In [ ]:
# Configs
out_folder = Path("../data/interim/sensitivity_test")
out_folder.mkdir(parents=True, exist_ok=True)

employment_cols = ['RETEMPN', 'FPSEMPN','HEREMPN', 'AGREMPN', 'MWTEMPN', 'OTHEMPN']
relevant_emp = ['RETEMPN', 'FPSEMPN', 'AGREMPN', 'MWTEMPN']
total_employment_col = 'TOTEMP'


projects = {
    "BridgeheadIndustrial": {"TAZ": 1182, "new_emp": 3500},
    "GiovannoniLogisticCenter": {"TAZ": 1292, "new_emp": 3643},
    "SuisunLogistics": {"TAZ": 1251, "new_emp": 2059},
}

In [ ]:
for project_name, p in projects.items():
    scenario_df = taz.copy()
    out_path = out_folder / f"tazData_{project_name}.dbf"

    taz_mask = scenario_df["ZONE"] == p["TAZ"]

    existing_emp = scenario_df.loc[taz_mask, relevant_emp].iloc[0]
    existing_total = existing_emp.sum()


    allocation = ((existing_emp / existing_total) * p["new_emp"]).round().astype(int)
    scenario_df.loc[taz_mask, relevant_emp] += allocation.values

    
    scenario_df[total_employment_col] = scenario_df[employment_cols].sum(axis = 1)

    print (f"Scenario: {project_name}")
    for col in employment_cols + [total_employment_col]:
        print (f"{col} Before: {taz.loc[taz["ZONE"] == p["TAZ"], col].values[0]}")
        print (f"{col} After: {scenario_df.loc[scenario_df["ZONE"] == p["TAZ"], col].values[0]}")
    print(f"Saving {project_name} in {out_path}")
    print ("")

    save_df_as_dbf(scenario_df, out_path, path)